# Warp: watch a match deform onto your query

Search as usual, pick one of the top results, and **watch it being warped onto your query object** as an animation. If both photos show the *same physical item*, details such as seams, creases and patterns slide into place and stop moving at the end. If it's a different item (even the same model), they don't line up.

**How the warp is made**
- Keypoints are matched between the query box and the result (the same matches stage 2 uses).
- From those matches, three warp models are fitted:
  - **H** (homography): a flat surface seen from another angle.
  - **A** (affine): rotation, scale and shear, without perspective.
  - **TPS** (thin-plate spline): a smooth, *bendable* warp through the matches, for fabric, cushions and small 3D effects. `TPS_STIFFNESS` trades exactness against smoothness.
- The animation starts with the result simply resized into your query box (t = 0) and ends with the full warp (t = 1). The motion in between is what the warp adds.

**Views**
- **morph:** the result deforms over your query, and the green dots are matches travelling to their place.
- **grid:** a grid drawn on the result bends along, showing where and how strongly the warp deforms.
- **flicker:** a blink between your query and the fully warped result. For the same item, details stay put.
- **residual:** a heatmap of the difference inside your query box, which should fade as the warp completes.

This notebook adds no new search logic. It uses the same cascade as `search.ipynb` and leaves the other notebooks unchanged.

## 1. Setup

This cell switches matplotlib to interactive widgets, imports the package, sets the knobs and loads DINOv2:
- `TOP_N`: how many results of a search you can pick from.
- `N_FRAMES`: animation length from unwarped to warped (the end state is held a little longer).
- `CANVAS_SIDE`: animation size in pixels.
- `TPS_STIFFNESS`: smoothness of the bendable warp. 0 passes exactly through every match; larger values ignore small match errors.
- `OUTPUT_DIR`: where **Save GIF** writes (`results/warps/`, gitignored).
- `GALLERY_SOURCE`:
  - `None` searches your labelled test set (results are your boxes);
  - a folder, glob or list of files searches those photos instead, with grounding finding the object in each photo (as in `search.ipynb`, section 4).

In [ ]:
%matplotlib widget
import logging
import os
from pathlib import Path

from IPython.display import Image, display

from visual_search.cascade import Cascade
from visual_search.data import AnnotationSet, load_gallery, make_synthetic_dataset, split_dataset
from visual_search.data.schema import parse_dataset
from visual_search.device import device_sync
from visual_search.masking import BoxMasker
from visual_search.models.dinov2 import Dinov2Backbone
from visual_search.retrieval import DinoRetriever, GroundedDinoRetriever
from visual_search.stages import SparseVerificationStage
from visual_search.ui import WarpApp
from visual_search.viz.animation import VIEWS, gif_bytes, save_gif, warp_frames

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
TESTSET_DIR = Path(os.environ.get("VS_TESTSET", ROOT / "data" / "testset"))
CACHE_DIR = ROOT / "cache"
OUTPUT_DIR = ROOT / "results" / "warps"

GALLERY_SOURCE = None  # None = your test set; or a folder / glob / list of photos
TOP_N = 10
N_FRAMES = 24
CANVAS_SIDE = 480
TPS_STIFFNESS = 0.05
DINO_MODEL = "facebook/dinov2-small"
DINO_RESOLUTION = 448
RETRIEVAL_K = 100
SPARSE_K = 20

backbone = Dinov2Backbone(DINO_MODEL, resolution=DINO_RESOLUTION)
sync = device_sync(backbone.device)
print("DINOv2 on", backbone.device)

## 2. Gallery and queries

This cell decides what is searched:
- **Test set** (the default, `GALLERY_SOURCE = None`): every labelled box is a gallery item, and the query dropdown offers those boxes. Without a test set, generated synthetic data is used instead.
- **A photo collection** (`GALLERY_SOURCE` set): every photo is a gallery item, and grounding finds where your object is in each photo. Queries come from the test-set dropdown or **Load path**.

In [ ]:
if (TESTSET_DIR / "annotations.json").exists():
    testset_items = parse_dataset(AnnotationSet.load(TESTSET_DIR).to_records())
    queries = [it for it in testset_items if it.label]
else:
    print(f"No annotations.json in {TESTSET_DIR}; using synthetic data.")
    synthetic = split_dataset(make_synthetic_dataset(n_instances=12, views_per_instance=3, n_distractors=40, seed=0), seed=0)
    testset_items, queries = synthetic.gallery + synthetic.queries, synthetic.queries

if GALLERY_SOURCE is None:
    gallery = testset_items
    retriever = DinoRetriever(backbone, pooling="crop_cls", cache_dir=CACHE_DIR)
else:
    gallery = parse_dataset(load_gallery(GALLERY_SOURCE))
    retriever = GroundedDinoRetriever(backbone, cache_dir=CACHE_DIR)
print(f"{len(gallery)} gallery items, {len(queries)} queries in the dropdown")

## 3. The warp app

This cell builds the app:
- **Top part:** the familiar search (query, box, **Search**, results per stage). It runs DINOv2 and then SIFT verification of the top `max(SPARSE_K, TOP_N)`.
- **Below it:** the warp panel.

The first run indexes the gallery, which takes a while once; after that it comes from `cache/`.

In [ ]:
cascade = Cascade(retriever, RETRIEVAL_K,
                  [SparseVerificationStage("sift", top_k=max(SPARSE_K, TOP_N), cache_dir=CACHE_DIR)], sync=sync)
app = WarpApp(cascade, gallery, queries, top_n=TOP_N, masker=BoxMasker(), output_dir=OUTPUT_DIR,
              n_frames=N_FRAMES, canvas_side=CANVAS_SIDE, tps_stiffness=TPS_STIFFNESS,
              show_expected_label=GALLERY_SOURCE is None)
app.widget

**How to use it**

1. **Search.** Pick a query (dropdown, or **Load path**), draw a box around the object and press **Search**.
2. **Pick a candidate** from the top `TOP_N` in the **Candidate** dropdown.
   - The pair appears **side by side**: your query box and the candidate's box or found region.
   - The line below says how many matches there are and which warps could be fitted.
3. **Choose a model and a view**, optionally *all models side by side*, and press **Warp ▶**.
   - The animation plays in a loop.
   - Use the play/pause button, drag the **frame** slider to scrub, and change **fps** for speed.
4. **Save GIF** writes the current animation to `results/warps/` (file name: query, candidate, model, view).

**Reading the table under the animation** (one row per model):
- `points`: matches used. More is better, but textured objects simply have more.
- `reproj_px`: how far the warped matches land from their partners (lower = tighter; TPS is near 0 by design).
- `strain`: how unevenly the warp stretches inside your box. 0 = rigid; large values mean strong local distortion, which often signals a wrong match.
- `residual`: the remaining appearance difference inside your box after warping (0–1, lower = more alike).

A candidate whose matches are too few for a model shows why under the pair, and that model can't be animated.

## 4. Export every view at once (optional)

After you've picked a candidate in the app, this cell renders all four views for the selected model and saves them as GIFs in `results/warps/`. It shows the morph inline (GIFs play in the notebook output). It does nothing until a candidate is selected.

In [ ]:
MODEL = "TPS"  # "TPS", "H" or "A"
if app.pair is not None and app.fitted is not None and MODEL in app.fitted.warps:
    query_id = app.search_app.searched_query.item_id
    for view in VIEWS:
        frames = warp_frames(app.pair, app.fitted.warps[MODEL], view, N_FRAMES, inliers=app.fitted.inliers.get(MODEL))
        path = save_gif(frames, OUTPUT_DIR / f"export_{view}_{MODEL}.gif")
        print("saved", path)
        if view == "morph":
            display(Image(data=gif_bytes(frames)))
else:
    print("Search and pick a candidate in the app first (and check that the model could be fitted).")